# Inpainting, Outpainting & Image Editing

Text-to-image makes new things. Inpainting fixes old ones. In production, 70% of billable image work is editing - swap a background, remove a logo, extend the canvas, regenerate a hand. Inpainting is where diffusion earns its keep.

## Problem definition

Variants of Inpainting:
1. Inpainting.   Regenerate inside a mask, keeo outside pixels
2. Outpainting.  Regenerate outside a mask (or beyond the canvas), keep inside.
3. Image Editing. Regenerate the whole image but keep semantic or strctural fidelity to the original.

## Basic Concept

Inpainting: mask-aware denoising with context-perserving reinjection.

### Navie apporach (and why it's wrong)

Run standard text-to-image with a mask. At each sampling step, replace the unmasked region of the noisy latent with the forward-duffused clean image. It works... badly. Boundary artifacts bleed through because the model has no information about what is in the masked region.

### The proper inpainting model

Train a modified U-Net that takes 9 input channels instead of 4:
```
input = concat(
    [noisy_latent (4ch), encoded_image (4ch), mask (1ch)],
    dim = channel
)
```
The extra channels are a copy of the VAE-encoded source image plus a single-channel mask. At the training time, you randomly mask regions of the image and train the model to denoise only the masked region while the unmasked region is given as a clean conditioning signal. At inference, the model can "see" what surrounds the masked region and produces coherent completions.

### SDEdit

Add noise to the source image up to some intermediate `t`, then run the reverse chain from `t` down to 0 with a new prompt. No retraining, the choice of starting `t` trades fidelity for creative freedom:

* `t/T = 0.3` --> Nearly identical to source ,small stylistic changes.
* `t/T = 0.6` --> moderate edits, preserves coarse structure.
* `t/T = 0.9` --> generated from near-noise, minimal source preservation

### InstructPix2Pix

Fine-tune a diffusion model on `(input_image, instruction, output_image)` triples. At inference, condition on both the input image and a text instruction. Two CFG scales: image scale and text scale.

### RePaint

Keep a standard unconditional diffusion model. At each reverse step, resample -- jump back to a noiser state occasionally and regenerate. Avoids bounary artifacts, Used when you don't have a trained inpainting model.

# Build your Own

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

# T=40 + beta_max=0.02 → alpha_bar[-1]≈0.67：终点还剩大量信号，
# 从纯噪声反推会和训练分布对不上。加大 T / beta，让终点接近纯噪声。
T, T_EMB_DIM, HIDDEN_DIM, REPRESENT_DIM = 100, 16, 64, 5
STEPS, BATCH, LR = 4000, 64, 1e-2


def make_schedule(steps):
    betas = torch.linspace(1e-4, 0.08, steps)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alphas, alpha_bars


class Denoiser(nn.Module):
    def __init__(self, input_dim, time_emb_dim, hidden_dim):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(input_dim + time_emb_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x, t_emb):
        return self.seq(torch.cat([x, t_emb], dim=-1))


def sin_embed(t, dim=T_EMB_DIM):
    """Sinusoidal timestep embedding. t: int or (B,) long."""
    if not torch.is_tensor(t):
        t = torch.tensor([t], dtype=torch.long)
    half = dim // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=t.device).float() / max(half - 1, 1)))
    angles = t.float().unsqueeze(1) * freqs
    return torch.cat([angles.sin(), angles.cos()], dim=-1)[:, :dim]


def sample_data(n):
    """5-D two-mode mixture: all dims ≈ ±1."""
    mode = torch.randint(0, 2, (n,))
    mean = torch.where(mode == 0, -1.0, 1.0)
    return mean.unsqueeze(-1).expand(-1, REPRESENT_DIM) + 0.2 * torch.randn(n, REPRESENT_DIM)


betas, alphas, alpha_bars = make_schedule(T)
print(f"schedule: T={T}, alpha_bar[-1]={alpha_bars[-1].item():.4f} (want ≪ 1)")

net = Denoiser(REPRESENT_DIM, T_EMB_DIM, HIDDEN_DIM)
opt = torch.optim.Adam(net.parameters(), lr=LR)
loss_fn = nn.MSELoss()


@torch.no_grad()
def ddpm_sample(n=400):
    x = torch.randn(n, REPRESENT_DIM)
    for t in range(T - 1, -1, -1):
        t_batch = torch.full((n,), t, dtype=torch.long)
        eps_hat = net(x, sin_embed(t_batch))
        mean = (x - betas[t] / (1.0 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        x = mean if t == 0 else mean + betas[t].sqrt() * torch.randn_like(x)
    return x


@torch.no_grad()
def naive_inpaint(x0, mask):
    """mask=1: regenerate; mask=0: keep. Reinjection each reverse step."""
    n = x0.shape[0]
    x = torch.randn_like(x0)
    for t in range(T - 1, -1, -1):
        eps = torch.randn_like(x0)
        x_known = alpha_bars[t].sqrt() * x0 + (1.0 - alpha_bars[t]).sqrt() * eps
        x = torch.where(mask.bool(), x, x_known)
        t_batch = torch.full((n,), t, dtype=torch.long)
        eps_hat = net(x, sin_embed(t_batch))
        mean = (x - betas[t] / (1.0 - alpha_bars[t]).sqrt() * eps_hat) / alphas[t].sqrt()
        x = mean if t == 0 else mean + betas[t].sqrt() * torch.randn_like(x)
    return torch.where(mask.bool(), x, x0)


########################## Training ##########################
losses = []
for step in range(1, STEPS + 1):
    x0 = sample_data(BATCH)
    t = torch.randint(0, T, (BATCH,))
    eps = torch.randn_like(x0)
    abar = alpha_bars[t].unsqueeze(-1)
    x_t = abar.sqrt() * x0 + (1.0 - abar).sqrt() * eps

    loss = loss_fn(net(x_t, sin_embed(t)), eps)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

    if step % 1000 == 0:
        print(f"Step {step}: Loss={loss.item():.4f}")

########################## Visualization ##########################
net.eval()
reals = sample_data(400)
gens = ddpm_sample(400)

# mask=1 → 洞(要重绘)；mask=0 → 保留。这里保留 d0–d2，重绘 d3–d4
x0 = sample_data(1)
mask = torch.tensor([[0.0, 0.0, 0.0, 1.0, 1.0]])  # 1 = inpaint hole
x_masked = x0.clone()
x_masked[mask.bool()] = 0.0  # 仅用于画图：把洞置 0
x_inpaint = naive_inpaint(x0, mask)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].plot(losses, alpha=0.35, lw=0.8, color="C0")
w = 100
ma = np.convolve(losses, np.ones(w) / w, mode="valid")
axes[0].plot(range(w - 1, len(losses)), ma, color="C0", lw=2, label=f"MA({w})")
axes[0].set_title("Training loss")
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE(eps)")
axes[0].legend(fontsize=8)

bins = np.linspace(-2.5, 2.5, 40)
axes[1].hist(reals.mean(dim=-1).numpy(), bins=bins, density=True, alpha=0.5, label="real")
axes[1].hist(gens.mean(dim=-1).numpy(), bins=bins, density=True, alpha=0.5, label="DDPM")
axes[1].axvline(-1, color="gray", ls="--", lw=1)
axes[1].axvline(1, color="gray", ls="--", lw=1)
axes[1].set_title("Sample mean across dims")
axes[1].set_xlabel("mean(x)")
axes[1].legend(fontsize=8)

idx = np.arange(REPRESENT_DIM)
wbar = 0.25
axes[2].bar(idx - wbar, x0.squeeze().numpy(), width=wbar, label="original")
axes[2].bar(idx, x_masked.squeeze().numpy(), width=wbar, label="input (hole=0)")
axes[2].bar(idx + wbar, x_inpaint.squeeze().numpy(), width=wbar, label="inpainted")
axes[2].axvspan(2.5, 4.5, color="red", alpha=0.12, label="hole (masked)")
axes[2].set_xticks(idx)
axes[2].set_xticklabels([f"d{i}
{'hole' if mask[0, i] else 'keep'}" for i in idx])
axes[2].set_ylim(-2.0, 2.0)
axes[2].set_title("Inpaint: keep d0–d2, fill d3–d4")
axes[2].legend(fontsize=7, loc="lower right")

plt.tight_layout()
plt.show()

err = (x_inpaint - x0).abs().squeeze()
print(
    "per-dim |inpaint - original|:",
    [f"{e:.3f}" for e in err.tolist()],
)
print(
    f"kept d0–d2 err = {err[:3].sum().item():.3f} (should be 0) | "
    f"hole d3–d4 err = {err[3:].mean().item():.3f} (expected > 0)"
)
